# Function 7: Sometimes Lazy is Best
You are now optimising six hyper-parameters of a machine learning model. Note that it is a popular and frequently used model, so maybe you could search to see if anyone else has optisized it before?

In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

In [2]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(7)
outputs_array = get_output_points(7)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))
y = y.ravel()

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (40, 6)
New shape of Y: (40,)


In [3]:
# Fit GP on the combined seed data and prior submissions.
kernel = C(1.0, (1e-3, 1e5)) * Matern(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(1e-2, 20.0),
    nu=2.5,
) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=20,
    normalize_y=True,
    random_state=42,
)
gp.fit(X, y)

def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mean, std = gp.predict(x, return_std=True)
    mean = mean.ravel()
    std = std.ravel() + 1e-9
    improvement = mean - y_best - xi
    z = improvement / std
    ei = improvement * norm.cdf(z) + std * norm.pdf(z)
    return -ei[0]

best_idx = np.argmax(y)
x_best = X[best_idx]
y_best = y[best_idx]

bounds = [(0.01, 0.99)] * X.shape[1]

# Blend local refinement around the current best point with a few global restarts.
rng = np.random.default_rng(42)
local_starts = np.clip(x_best + rng.uniform(-0.05, 0.05, size=(25, X.shape[1])), 0.01, 0.99)
global_starts = rng.uniform(0.01, 0.99, size=(10, X.shape[1]))
initial_points = np.vstack([local_starts, global_starts])

best_x = None
best_score = float("inf")
for x0 in initial_points:
    result = minimize(expected_improvement, x0=x0, bounds=bounds, args=(gp, y_best), method="L-BFGS-B")
    if result.fun < best_score:
        best_score = result.fun
        best_x = result.x

next_query = np.round(best_x, 6)
formatted_next_query = f"{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}-{next_query[5]:.6f}"
print("Next Query Point:", formatted_next_query)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Next Query Point: 0.010000-0.366272-0.162337-0.105428-0.376024-0.760802
